In [47]:
from dotenv import load_dotenv
load_dotenv()

True

In [48]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [49]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [50]:
vectorstore

In [51]:
retriever = vectorstore.as_retriever()

In [52]:
from langchain_core.prompts import PromptTemplate

rag_prompt = PromptTemplate.from_template(
    """당신은 정보 검색 전문가입니다.
다음 컨텍스트를 분석하여 사용자의 질문에 답하기 위해 필요한 모든 정보를 순서가 없는 목록 형식(-)으로 제공하시오.
문장 형태의 완결된 답변이나 인사말은 생략하고 정보만 전달하세요.

컨텍스트: {context}
질문: {question}
"""
)

In [53]:
from langchain.chat_models import init_chat_model

rag_model = init_chat_model("google_genai:gemini-3-flash-preview")

In [54]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()} 
    | rag_prompt 
    | rag_model 
    | StrOutputParser()
)

In [55]:
from langchain.tools import tool

@tool
def ask_knowledge_expert(query: str) -> str:
    """테크노빌드 주식회사 사내 가이드북에 대한 전문적인 지식이 필요할 때 사용합니다.
    검색 결과를 바탕으로 정제된 답변을 제공합니다."""
    return rag_chain.invoke(query)

In [56]:
system_prompt = """당신은 테크노빌드 주식회사 가이드북 정보를 친절하게 제공하는 어시스턴트입니다.

1. 정보가 필요할 경우 반드시 검색 도구(ask_knowledge_expert)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 추측하지 말고 모른다고 답변하세요.
"""

In [57]:
from langchain.agents import create_agent

main_agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[ask_knowledge_expert],
    system_prompt=system_prompt
)

In [58]:
from langchain.messages import HumanMessage

query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

response = main_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

response

{'messages': [HumanMessage(content='국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?', additional_kwargs={}, response_metadata={}, id='7d40faf4-3d35-46bd-9b01-437bf6253a7b'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ask_knowledge_expert', 'arguments': '{"query": "\\uae30\\uc0ac \\uc790\\uaca9\\uc99d \\ucde8\\ub4dd \\uc2dc \\uc218\\ub2f9"}'}, '__gemini_function_call_thought_signatures__': {'df7762bb-4bfe-4018-92f7-bebbf281febb': 'EjQKMgEMOdbHGW2rJTNWo7UWyhXD+A9AZUauoWnSB7alu7DeC3tiz5gfyoOL2++DelrfpwZu'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e3473-b30c-7573-92ba-150e22243702-0', tool_calls=[{'name': 'ask_knowledge_expert', 'args': {'query': '기사 자격증 취득 시 수당'}, 'id': 'df7762bb-4bfe-4018-92f7-bebbf281febb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 190, 'output_tokens': 27, 'total_tokens': 217, 'input_token_details': {'c

In [60]:
print(response['messages'][2].content)

- 자격 등급: 기사
- 축하금(1회성): 50만 원
- 자격 수당(월): 10만 원
- 대상 자격증 예시: 일반기계, 전기, 산업안전 등
- 수당 인정 조건: 동일 등급 내 1개 자격증만 인정 (상위 등급 취득 시 갱신)
- 축하금 지급 조건: 횟수 제한 없음
- 축하금 지급 시기: 자격증 제출 후 2주 이내 별도 입금
- 수당 확인 방법: 급여 명세서의 ‘자격 수당’ 항목에서 확인 가능
